# Estrazione episodi completi dal debug dataset

**Scopo.** A partire da `data_for_training/debug_dataset.pkl` (che è una **lista piatta di `Transition`**, cioè transizioni singole non raggruppate) ricostruire gli **episodi completi** e salvarli in un file **nuovo** `debug_dataset_full_ep.pkl`, **senza mai sovrascrivere** l'originale.

**Struttura verificata del sorgente** (caricando realmente il pickle):
- `debug_dataset.pkl` → `list[Transition]`, 27143 transizioni totali.
- Ogni `Transition` ha: `observation` (16,), `action` (2,), `true_reward` (float), `next_status` (one-hot 7-dim `[arrived, collided, off_road, timeout, running, teleported, removed]`), `done` (bool).
- Il flag `done==True` coincide **esattamente** con la transizione terminale (status non-running): 80 transizioni con `done=True` ⇒ **80 episodi**, contigui, senza coda orfana finale.
- Distribuzione terminazioni: 20 arrived / 20 collided / 20 off_road / 20 timeout (dataset bilanciato).

**Criterio di split.** Un episodio = sequenza contigua di transizioni che termina nella prima `Transition` con `done==True` (inclusa). L'output è una `list[Trajectory]` per essere coerente con `expert_trajectories.pkl` (anch'esso `list[Trajectory]`) e con le funzioni di plotting che iterano `for t in trajectory`.

> Nota: questo notebook va eseguito nell'ambiente in cui è installato il package `human_feedback_rl` (Python 3.10), così che `Trajectory`/`Transition` siano la classe reale e il pickle risultante sia ricaricabile altrove.

In [1]:
import pickle
from pathlib import Path
from collections import Counter

import numpy as np

from human_feedback_rl.common.types import Trajectory, Transition

# Il notebook vive in <repo>/notebooks/. Risaliamo a <repo>/data_for_training.
# Se esegui da un'altra working dir, modifica DATA_DIR di conseguenza.
DATA_DIR = Path.cwd().parent / "data_for_training" if Path.cwd().name == "notebooks" else Path("data_for_training")
SRC = DATA_DIR / "debug_dataset.pkl"
DST = DATA_DIR / "debug_dataset_full_ep.pkl"

assert SRC.exists(), f"Sorgente non trovato: {SRC.resolve()}"
print("SRC:", SRC.resolve())
print("DST:", DST.resolve())

SRC: /Users/andreazhang/Desktop/newTesi/sumo-human-feedback-rl/data_for_training/debug_dataset.pkl
DST: /Users/andreazhang/Desktop/newTesi/sumo-human-feedback-rl/data_for_training/debug_dataset_full_ep.pkl


In [2]:
# --- Carico e verifico che il sorgente sia davvero una lista piatta di Transition ---
with open(SRC, "rb") as f:
    transitions = pickle.load(f)

assert isinstance(transitions, list), f"Atteso list, trovato {type(transitions)}"
assert all(isinstance(t, Transition) for t in transitions[:50]), "Gli elementi non sono Transition"

n_done = sum(bool(t.done) for t in transitions)
print(f"Transizioni totali: {len(transitions)}")
print(f"Transizioni con done==True (= numero atteso di episodi): {n_done}")
print(f"Ultima transizione ha done==True? {bool(transitions[-1].done)}")

Transizioni totali: 27143
Transizioni con done==True (= numero atteso di episodi): 80
Ultima transizione ha done==True? True


In [3]:
# --- Split in episodi completi: chiudo un episodio sulla prima transizione con done==True ---
def split_into_episodes(transitions):
    episodes, current = [], []
    for t in transitions:
        current.append(t)
        if bool(t.done):
            episodes.append(Trajectory(current))
            current = []
    if current:  # eventuale coda senza done finale: la segnalo e la salvo come episodio incompleto
        print(f"ATTENZIONE: {len(current)} transizioni finali senza done==True "
              f"(episodio incompleto, appeso comunque in coda).")
        episodes.append(Trajectory(current))
    return episodes

episodes = split_into_episodes(transitions)
print(f"Episodi estratti: {len(episodes)}")

Episodi estratti: 80


In [4]:
# --- Sanity check: nessuna transizione persa/duplicata, lunghezze e bilanciamento ---
STATUS = {0: "arrived", 1: "collided", 2: "off_road", 3: "timeout",
          4: "running", 5: "teleported", 6: "removed"}

assert sum(len(ep) for ep in episodes) == len(transitions), "Conteggio transizioni non conservato!"
assert all(isinstance(ep, Trajectory) for ep in episodes)
assert all(len(ep) > 0 for ep in episodes), "Episodio vuoto trovato"

lens = [len(ep) for ep in episodes]
term = [int(np.argmax(ep[-1].next_status)) for ep in episodes]
n_terminated = sum(bool(ep[-1].done) for ep in episodes)

print(f"Episodi: {len(episodes)} | transizioni totali (conservate): {sum(lens)}")
print(f"Lunghezze episodi -> min: {min(lens)}  mean: {np.mean(lens):.1f}  max: {max(lens)}")
print(f"Episodi che terminano con done==True: {n_terminated}/{len(episodes)}")
print("Distribuzione status terminale:",
      {STATUS[k]: v for k, v in sorted(Counter(term).items())})

Episodi: 80 | transizioni totali (conservate): 27143
Lunghezze episodi -> min: 4  mean: 339.3  max: 600
Episodi che terminano con done==True: 80/80
Distribuzione status terminale: {'arrived': 20, 'collided': 20, 'off_road': 20, 'timeout': 20}


In [5]:
# --- Salvataggio SICURO: non sovrascrive l'originale e rifiuta di sovrascrivere un DST esistente ---
assert SRC.resolve() != DST.resolve(), "DST coincide con SRC: operazione bloccata."
if DST.exists():
    raise FileExistsError(
        f"{DST} esiste già. Rimuovilo manualmente se vuoi rigenerarlo "
        f"(blocco intenzionale per evitare sovrascritture accidentali)."
    )

with open(DST, "wb") as f:
    pickle.dump(episodes, f)

print(f"Salvati {len(episodes)} episodi in: {DST.resolve()}")
print(f"Originale intatto: {SRC.resolve()} ({SRC.stat().st_size} bytes)")

Salvati 80 episodi in: /Users/andreazhang/Desktop/newTesi/sumo-human-feedback-rl/data_for_training/debug_dataset_full_ep.pkl
Originale intatto: /Users/andreazhang/Desktop/newTesi/sumo-human-feedback-rl/data_for_training/debug_dataset.pkl (6190262 bytes)
